In [1]:
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
import os
import openai

openai.api_key = config('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = openai.api_key

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
from typing import List, Tuple

def load_doc_from_urls(urls:List[str],
                       tags:List[str],
                       tag_classes:List[str]):
    urls = tuple(urls)
    tag = tuple(tags)
    tag_classes = tuple(tag_classes)

    # Load, chunk and index the contents of the blog.
    loader = WebBaseLoader(
        web_paths=urls,
        bs_kwargs=dict(
            parse_only=bs4.SoupStrainer(tag,
                class_=tag_classes
            )
        ),
    )

    docs = loader.load()
    return docs

In [4]:
from pydantic import BaseModel, Field
class Overview(BaseModel):
    """Overview of a section of text."""
    summary: str = Field(description="Provide a concise summary of the content.")
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content.")

In [5]:
from langchain.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not explicitly provided do not guess. Extract partial info"),
    ("human", "{input}")
])

In [6]:
from langchain_openai import ChatOpenAI
model_name = 'gpt-4o-mini'
llm = ChatOpenAI(
    model=model_name,
    temperature=0.0,
    max_tokens=1024
)

llm

In [ ]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser
from langchain.utils.openai_functions import convert_pydantic_to_openai_function

overview_tagging_function = [
    convert_pydantic_to_openai_function(Overview)
]
tagging_model = model.bind(
    functions=overview_tagging_function,
    function_call={"name":"Overview"}
)
tagging_chain = prompt | tagging_model | JsonOutputFunctionsParser()

In [29]:
docs = load_doc_from_urls(
    urls=['https://www.moneycontrol.com/news/business/economy/'],
    tags=['a',],
    tag_classes=[] 
)

In [ ]:
docs

In [ ]:
tagging_chain.invoke({"input": docs})

In [41]:
from typing import List

class News(BaseModel):
    """Information about website mentioned."""
    news_summary: str = Field(description="News mentioned in text extracted from the website")
    sentiment: str = Field(description="The sentiment associated with the news.")
    entity: str = Field(description="Important named entities mentioned in the news, if any")


class Info(BaseModel):
    """Information to extract"""
    news: List[News]

In [42]:
template = """A news webpage will be passed to you. Extract the news that are mentioned in this article.
If not clear, do not gues or extract partial info"""

prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", "{input}")
])

In [43]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

tip_extraction_function = [
    convert_pydantic_to_openai_function(Info)
]
extraction_model = llm.bind(
    functions=tip_extraction_function, 
    function_call={"name":"Info"}
)
extraction_chain = prompt | extraction_model | JsonOutputFunctionsParser()


In [ ]:
result = extraction_chain.invoke({"input": docs[0]})
print(result)

In [ ]:
type(result['news'][0])

In [ ]:
for news in result['news']:
    for key, val in news.items():
        print(f"{key}: {val}")
    print("-----")